# FastPtx demo
This demo demonstrates how FastPtx can be used to optimize free-waveform RF and gradient pulses that can be used with a pTx system. The toolbox is described in detail in the corresponding paper:

> Bosch, D., & Scheffler, K. (2024). FastPtx: a versatile toolbox for rapid, joint design of pTx RF and gradient pulses using Pytorch's autodifferentiation. Magma (New York, N.Y.), 37(1), 127–138. https://doi.org/10.1007/s10334-023-01134-7

The demo uses only a subset of the features FastPtx provides. It will allow you to calculate a free-waveform pTx pulse in only a few minutes using on Google Colab.

## Install requirements and download example data
The dataset contains B1+ and B0 maps that were acquired on a 9.4 Tesla human MRI scanner with a coil with 16 RF channels. It is a subset of the dataset that was used for the original paper. The full dataset cana lso be used by changing the download command below.

In [ ]:
import os, sys
IN_ORIGINAL_REPO = os.path.isdir('tools') and os.path.isfile('tools/smallTipAngle.py')
# Download example data
if not os.path.isfile('data.tar'):
  # full dataset
  #!wget "https://keeper.mpdl.mpg.de/f/432e8e08fb284730b357/?dl=1" -O data.tar
  # reduced dataset
  !wget "https://keeper.mpdl.mpg.de/f/b2b993681f594889a84a/?dl=1" -O data.tar
if not os.path.isdir('data'):
  !tar -xf data.tar
# If we're in the original repo, we only need to install the requirements. Otherwise first clone the FastPtx code.
if IN_ORIGINAL_REPO:
  !pip install -r requirements.txt
else:
  if not os.path.isdir('FastPtx'):
    !git clone -b dev https://github.com/dabosch/FastPtx
  !pip install -r FastPtx/requirements.txt
  


## Load requirements

In [ ]:
if IN_COLAB:
    sys.path.append('./FastPtx')
from  tools.plot3D import plot3D
from tools.io import readData, combineData, vectorizeData, writePulse, readVOPfile, interpData
from tools.smallTipAngle import smallTipAngle
from matplotlib import pyplot as plt
import torch
from scipy.io import savemat


# Adapt settings
Here we load default settings and adapt those that are of interest to us.
We also 

In [ ]:
# load one set of B1/B0 maps.
fname = ['./data/EBNY-7PGC.mat']
# Alternatively, multiple sets can be loaded to calculate a universal pulse.
# Be careful since you might eventually run out of memory, especially when working on a GPU
# fname = ['./data/EBNY-7PGC.mat','./data/DJOY-ISET.mat', './data/4VRE-G6Q4']

# Which VOPs to use for SAR calculation. Optional. Instead of loading VOPs you can set them to None when calling the optimizer
vopfile = './data/RFSWDZZMat9001.bin.mat'
downsample=False # downsample for faster calculation - use only for demo/development

# system limits (SAR, maxRF, etc)
limits = smallTipAngle.getLimits()
limits['targetFA'] = 10.0 # desired FA in degrees
limits['plotscale'] = 10 * 1.2 # color scale for plot
limits['maxSAR'] = 2.3 # SAR limit [W/kg]
limits['maxRF'] = 100 # max RF voltage. 185 at 9.4T, probably similar at 7T
limits['plotiter'] = 400 # plot every 400 iterations
limits['dispiter'] = 40 # display info every 40 iterations

# how long (how many samples) should your pulse be? The first and last sample is forced to be 0 to ensure
# slew rate limits are kept
nSamples = 30
raster_time = 1e-5 # seconds

if torch.cuda.is_available():
    dev = torch.device('cuda') # 'cuda' or 'cpu'
else:
    dev = torch.device('cpu') # 'cuda' or 'cpu'
dtc = torch.complex64 # datatype to use for complex data
dt = torch.ones(1,dtype=dtc).abs().dtype # automatically determine datatype for real valued data

## Inspect one dataset
Before we start our optimization, let's have a look at the data. First we look at one of the original Matlab files.

In [ ]:
# load original dataset
from scipy.io import loadmat
m = loadmat(fname[0])
print(f"{m.keys()}")

## Read Data
Now we read the data using the readData function FastPtx provides, which performs some data conversion.

We also read the VOP file.

In [ ]:
# read data into a list
dat = []
for f in fname:
    dat.append(readData(f,dt=dt,dev=dev))
# if we have multiple datasets, they are now combined into one large one
datComb = dat[0];
for i in range(1,len(dat)):
    datComb = combineData(datComb,dat[i])
if downsample:
    datComb = interpData(datComb,0.5)
# Vectorize data (from 3D into 1D) and omit all voxels not part of the mask
dat_v = vectorizeData(datComb)

# read the VOP file
VOPs = readVOPfile(vopfile,dev=dev,dtc=dtc)

# set 3D target FA distribution to a homogeneous flip angle 
target = limits['targetFA'] * dat_v['mask3D'].clone()

## Start from CP mode pulse
For faster convergence and to provide a fair comparison we start from a CP mode pulse  

In [ ]:
# create CP mode pulse
nCh = dat_v['s'].shape[1]
start_pulse = torch.ones(nSamples,dat_v['s'].shape[1],dtype=dtc) * 20
start_pulse *= torch.exp(1j * torch.linspace(0,2*torch.pi,nCh+1)[:-1].reshape([1,-1]) )
start_pulse = start_pulse.to(dev)

## Run the optimizer
First make sure the img directory exists and is empty. Here some graphs will be stored documenting the optimization process.

Then run the optimizer for RF and gradient waveforms, for 2400 iterations.

In [ ]:
if os.path.isdir('img'):
  !rm -r ./img
!mkdir -p ./img

pulse,g = smallTipAngle.optimize_RF_grad(dat_v,target,niter=2400,nsteps=30,pulse_start=start_pulse,
                           g_start=None,stepdur=raster_time,VOPs=VOPs,TR=20e-3,limits=limits,do_plot=True)

## Save pulse
Save both the RF and gradient pulse into a .mat file, as well as into an .ini file as required for Siemens scanners.

In [ ]:
# %% export pulse file
mat = dict()
mat['pulse'] = pulse.cpu().numpy()
mat['g'] = g.cpu().numpy()
mat['stepdur'] = raster_time
orientation='unity'
writePulse('opt_result.ini', pulse.cpu(), g.cpu(),orientation=orientation,stepdur=raster_time,FA=limits['targetFA'])
print('writing mat')
savemat('opt_result.mat',mat)